Before running this notbook, please run codonaligner.ipynb

In [1]:
import cogent3
from cogent3 import get_app
from cogent3.maths.matrix_exponential_integration import expected_number_subs
import matplotlib.pyplot as plt
import paths
import libs
import pandas as pd
import numpy as np
import pickle
from tqdm.notebook import tqdm

import trinuc_models as trinucs # this module must be in the same directory as this notebook

## CDS

In [23]:
#Setting up the rules for model fitting of cds regions
sm_noncds=trinucs.GNC_CpG_ss()
paramnames = sm_noncds.get_param_list()
rules_cds = [{"par_name": n, "is_independent": True} for n in paramnames]
GNC_subsmodel = get_app("model", "GNC_CpG_ss",
                      show_progress=True,
                      optimise_motif_probs=False,
                      param_rules=rules_cds)

#Setting up the app for reading cds sequences
loader_cds = get_app("load_aligned", moltype="dna")   
omit_degs_cds = get_app("omit_degenerates", moltype="dna", motif_length=3)
concat = get_app("concat", moltype="dna")

#create a concatenated alignment with all coding positions
cds_process = loader_cds + omit_degs_cds

In [24]:
folder_in = paths.DATA_APES114 + 'cds/codon_aligned'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

nonconcat_cds = [r for r in cds_process.as_completed(in_dstore[:], parallel=True) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

cds_alns = concat(nonconcat_cds)
cds_alns.source = "cds_alignments"

result_cds = GNC_subsmodel(cds_alns)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

with open("fitted_models/likelihood_cds.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_cds))

result_cds.lf

   0%|          |00:00<?

   0%|          |00:00<?

GNC_CpG_ss
log-likelihood = -779195.6887
number of free parameters = 102
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        0.01               4.48    1.13    3.66    0.68
Chimpanzee    root        0.02               3.60    0.97    1.93    0.83
Gorilla       root        0.03               5.06    1.19    2.95    0.87
-------------------------------------------------------------------------

continued: 
=====================================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C    omega
---------------------------------------------------------------------
1.22    1.57    4.45    3.66    1.36    1.15    0.83    2.83     0.35
0.79    1.12    2.17    1.87    1.33    0.94    0.64    1.94     0.45
1.00    1.47    2.90    2.45    1.55    1.31    0.68    2.94     0.33
---------------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.02    0.02    0.04    0.01    0.01    0.02    0.01    0.01    0.01    0.03
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.01    0.01    0.00    0.02    0.02    0.01    0.01    0.02    0.04    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.02    0.03    0.01    0.02    0.01    0.02    0.02    0.00    0.00    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.05    0.01    0.02    0.03    0.05    0.02    0.01    0.04    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAC     TAT
----------------------------------------------------------------------------
0.01    0.03    0.02    0.01    0.00    0.02    0.03    0.01    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TCA     TCC     TCG     TCT     TGC     TGG     TGT     TTA     TTC     TTG
----------------------------------------------------------------------------
0.01    0.02    0.01    0.01    0.01    0.01    0.01    0.00    0.02    0.01
----------------------------------------------------------------------------

continued: 
====
 TTT
----
0.01
----

## Intergenic Ancestral Repeats (IGAR) with codon substitution model

In [21]:
#Setting up the app for reading noncds sequences
#rename_noncds eliminates files that do not contain Gorilla, Human and Chimp sequences or that has duplicates.
loader = get_app("load_aligned", moltype="dna")
rename_noncds = libs.renamer_noncds_aligned()
#Because I am using a trinucleotide I need to omit degenerates using motif 3
omit_degs = get_app("omit_degenerates", moltype="dna", motif_length=3)

#trim_stop_codons only remove codons at the end of the sequence, I need to make a custome app that removes stop codons instead
trim_stops = get_app("trim_stop_codons", moltype="dna")

concat = get_app("concat", moltype="dna")

noncds_usingcodon_app = loader + rename_noncds + omit_degs + trim_stops

folder_in = paths.DATA_APES114 + 'intergenicAR/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

nonconcat_IGARusingcodon = [r for r in noncds_usingcodon_app.as_completed(in_dstore[:], parallel=True) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_IGARusingcodon = concat(nonconcat_IGARusingcodon)
alns_IGARusingcodon.source = "IGAR_alignments_usingcodon"

result_IGAR_usingcodon = GNC_subsmodel(alns_IGARusingcodon)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

with open("fitted_models/likelihood_IGAR_usingcodonmodel.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_IGAR_usingcodon))

result_IGAR_usingcodon.lf

   0%|          |00:00<?

AttributeError: 'NotCompleted' object has no attribute 'lf'

In [22]:
result_IGAR_usingcodon

NotCompleted(type=ERROR, origin=model, source="IGAR_alignments_usingcodon", message="Traceback (most recent call last):
  File "/home/u12/uliseshmc/.conda/envs/delme/lib/python3.13/site-packages/cogent3/evolve/likelihood_tree.py", line 261, in make_likelihood_tree_leaf
    likelihoods = get_matched_array(alphabet, moltype, uniq_motifs, dtype=float)
  File "/home/u12/uliseshmc/.conda/envs/delme/lib/python3.13/site-packages/cogent3/evolve/likelihood_tree.py", line 218, in get_matched_array
    for motif in moltype.resolve_ambiguity(ambig_motif, alphabet=alphabet):
                 ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/u12/uliseshmc/.conda/envs/delme/lib/python3.13/site-packages/cogent3/core/moltype.py", line 1034, in resolve_ambiguity
    raise c3_alphabet.AlphabetError(ambig_motif)
cogent3.core.alphabet.AlphabetError: TAA

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/u12/uliseshmc

# Non cds

In [4]:
#Setting up the rules for model fitting of noncds regions
sm_noncds=trinucs.GT_CpG_ss()
paramnames = sm_noncds.get_param_list()
rules_noncds = [{"par_name": n, "is_independent": True} for n in paramnames if n!="omega"] + [{"par_name": "omega", "value": 1.0, "is_constant": True}]
GT_subsmodel = get_app("model", "GT_CpG_ss",
                      show_progress=True,
                      optimise_motif_probs=False,
                      param_rules=rules_noncds)

#Setting up the app for reading noncds sequences
#rename_noncds eliminates files that do not contain Gorilla, Human and Chimp sequences or that has duplicates.
loader = get_app("load_aligned", moltype="dna")
rename_noncds = libs.renamer_noncds_aligned()
#Because I am using a trinucleotide I need to omit degenerates using motif 3
omit_degs_noncds = get_app("omit_degenerates", moltype="dna", motif_length=3)
concat = get_app("concat", moltype="dna")

noncds_app = loader + rename_noncds + omit_degs_noncds

## Intergenic Ancestral Repeats (IGAR)

In [5]:
folder_in = paths.DATA_APES114 + 'intergenicAR/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

nonconcat_IGAR = [r for r in noncds_app.as_completed(in_dstore[:], parallel=True) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_IGAR = concat(nonconcat_IGAR)
alns_IGAR.source = "IGAR_alignments"

result_IGAR = GT_subsmodel(alns_IGAR)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

with open("fitted_models/likelihood_IGAR.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_IGAR))

result_IGAR.lf

   0%|          |00:00<?

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -3470472.6747
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        0.02              10.28    2.49    5.86    3.36
Chimpanzee    root        0.03               9.53    1.73    3.26    1.99
Gorilla       root        0.04              11.66    0.84    3.42    0.80
-------------------------------------------------------------------------

continued: 
=============================================================
 C>A     C>G     C>T      G>A     G>C     G>T     T>A     T>C
-------------------------------------------------------------
2.57    3.32    7.83    10.31    0.00    0.00    0.00    5.81
0.00    2.98    4.69     7.29    0.00    0.00    0.00    3.27
0.97    1.91    3.81     3.88    1.14    1.06    0.60    3.25
-------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.03    0.01    0.02    0.02    0.02    0.01    0.00    0.02    0.02    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.02    0.02    0.01    0.01    0.02    0.02    0.02    0.02    0.03    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.02    0.02    0.00    0.02    0.00    0.00    0.00    0.00    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.03    0.02    0.02    0.01    0.02    0.01    0.02    0.02    0.00    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.02    0.02    0.02    0.01    0.01    0.01    0.02    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.01    0.02    0.02    0.00    0.02    0.02    0.02    0.02    0.02
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.01    0.02    0.02    0.03
----------------------------

## Introns

In [ ]:
folder_in = paths.DATA_APES114 + 'introns/noUTRs/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_introns = [r for r in noncds_app.as_completed(in_dstore[:], parallel=True) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_introns = concat(nonconcat_introns)
alns_introns.source = "introns_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_introns = GT_subsmodel(alns_introns)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

with open("fitted_models/likelihood_introns.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_introns))

result_introns.lf

   0%|          |00:00<?

   0%|          |00:00<?

## 5' UTR

In [7]:
folder_in = paths.DATA_APES114 + 'introns/5UTR/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_5UTR = [r for r in noncds_app.as_completed(in_dstore[:], parallel=True) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_5UTR = concat(nonconcat_5UTR)
alns_5UTR.source = "5UTR_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_5UTR = GT_subsmodel(alns_5UTR)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

with open("fitted_models/likelihood_5UTR.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_5UTR))

result_5UTR.lf

GT_CpG_ss
log-likelihood = -152226.8791
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        0.02               4.69    0.78    3.31    0.39
Chimpanzee    root        0.02               5.94    0.73    4.02    0.73
Gorilla       root        0.03               6.06    1.21    4.50    0.25
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
1.30    1.24    4.76    4.94    1.67    0.98    0.46    4.17
1.09    1.98    5.81    5.21    1.85    1.53    0.97    3.30
1.28    1.92    4.28    4.82    1.41    1.37    0.88    4.75
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.02    0.01    0.02    0.01    0.02    0.01    0.01    0.01    0.02    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.03    0.01    0.01    0.01    0.01    0.01    0.01    0.02    0.03    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.02    0.03    0.01    0.02    0.01    0.01    0.01    0.01    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.03    0.02    0.02    0.01    0.02    0.01    0.02    0.02    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.02    0.03    0.03    0.02    0.01    0.01    0.02    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.02    0.00    0.02    0.02    0.02    0.02    0.01
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.01    0.01    0.01    0.02
----------------------------

## 3' UTR

In [8]:
folder_in = paths.DATA_APES114 + 'introns/3UTR/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_3UTR = [r for r in noncds_app.as_completed(in_dstore[:], parallel=True) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_3UTR = concat(nonconcat_3UTR)
alns_3UTR.source = "3UTR_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_3UTR = GT_subsmodel(alns_3UTR)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

with open("fitted_models/likelihood_3UTR.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_3UTR))

result_3UTR.lf

   0%|          |00:00<?

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -143814.9454
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        0.02               5.26    0.99    5.17    0.31
Chimpanzee    root        0.02               6.15    1.73    6.13    0.89
Gorilla       root        0.03               5.21    0.85    3.15    0.43
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.90    1.70    5.17    4.94    1.49    0.73    0.44    4.13
1.21    3.25    9.41    7.48    2.07    2.20    1.02    5.47
0.65    1.42    3.34    3.16    1.16    0.71    0.46    3.06
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.02    0.01    0.02    0.01    0.01    0.02    0.01    0.01    0.02    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.03    0.01    0.01    0.01    0.01    0.01    0.01    0.02    0.03    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.02    0.03    0.01    0.03    0.00    0.01    0.01    0.01    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.03    0.02    0.02    0.01    0.02    0.01    0.02    0.03    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.02    0.03    0.03    0.01    0.01    0.01    0.02    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.01    0.01    0.02    0.00    0.02    0.02    0.02    0.02    0.02
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.01    0.02    0.01    0.02
----------------------------

## Introns Ancestral Repeats (IntronAR)

In [9]:
folder_in = paths.DATA_APES114 + 'intronsAR/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_intronsAR = [r for r in noncds_app.as_completed(in_dstore[:], parallel=True) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_intronsAR = concat(nonconcat_intronsAR)
alns_intronsAR.source = "intronsAR_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_intronsAR = GT_subsmodel(alns_intronsAR)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

with open("fitted_models/likelihood_intronsAR.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_intronsAR))

result_intronsAR.lf

   0%|          |00:00<?

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -8028260.6009
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        0.02              12.04    0.96    2.95    0.71
Chimpanzee    root        0.02               8.65    1.82    3.50    1.76
Gorilla       root        0.04              11.19    0.98    3.96    0.76
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
0.97    1.67    4.24    4.40    1.11    1.02    0.56    3.02
0.00    2.39    5.80    8.70    0.00    0.00    0.00    3.37
1.07    1.58    4.22    4.37    1.28    1.02    0.66    3.87
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.03    0.01    0.02    0.02    0.02    0.01    0.00    0.02    0.02    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.02    0.02    0.01    0.01    0.01    0.02    0.02    0.02    0.03    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.02    0.02    0.01    0.02    0.00    0.01    0.01    0.00    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.03    0.02    0.01    0.01    0.02    0.01    0.02    0.02    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.02    0.02    0.02    0.01    0.01    0.01    0.02    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.01    0.02    0.02    0.00    0.02    0.02    0.02    0.02    0.02
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.01    0.01    0.02    0.03
----------------------------

## Intergenic distal (5kb distal from transcript)

In [10]:
folder_in = paths.DATA_APES114 + 'distal_IG/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_distalIG = [r for r in noncds_app.as_completed(in_dstore[:], parallel=True) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_distalIG = concat(nonconcat_distalIG)
alns_distalIG.source = "distalIG_alignments"

result_distalIG = GT_subsmodel(alns_distalIG)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

with open("fitted_models/likelihood_distalIG.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_distalIG))

result_distalIG.lf

   0%|          |00:00<?

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -1584682.7479
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        0.02              10.87    1.12    4.09    0.68
Chimpanzee    root        0.02              10.97    1.06    3.75    0.73
Gorilla       root        0.04              10.74    0.96    4.33    0.67
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
1.33    1.65    4.99    5.27    1.53    1.39    0.58    4.01
1.22    1.60    4.63    4.55    1.46    1.29    0.75    3.51
1.24    1.65    4.75    4.72    1.45    1.12    0.65    4.41
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.02    0.01    0.02    0.02    0.02    0.01    0.00    0.01    0.02    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.03    0.01    0.01    0.01    0.02    0.01    0.02    0.02    0.03    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.02    0.03    0.01    0.03    0.00    0.00    0.01    0.00    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.03    0.02    0.02    0.01    0.02    0.01    0.02    0.02    0.00    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.02    0.02    0.02    0.01    0.01    0.01    0.02    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.01    0.02    0.02    0.00    0.02    0.02    0.02    0.02    0.02
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.01    0.02    0.01    0.02
----------------------------

## Intergenic proximal 5' (5kb proximal from transcript start)

In [11]:
folder_in = paths.DATA_APES114 + 'proximal5_IG/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_proximal5IG = [r for r in noncds_app.as_completed(in_dstore[:], parallel=True) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_proximal5IG = concat(nonconcat_proximal5IG)
alns_proximal5IG.source = "proximal5IG_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_proximal5IG = GT_subsmodel(alns_proximal5IG)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

with open("fitted_models/likelihood_proximal5IG.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_proximal5IG))

result_proximal5IG.lf

   0%|          |00:00<?

Exception ignored in: <function tqdm.__del__ at 0x7f40bc29e7a0>
Traceback (most recent call last):
  File "/home/u12/uliseshmc/.conda/envs/delme/lib/python3.13/site-packages/tqdm/std.py", line 1148, in __del__
  File "/home/u12/uliseshmc/.conda/envs/delme/lib/python3.13/site-packages/tqdm/notebook.py", line 282, in close
  File "/home/u12/uliseshmc/.conda/envs/delme/lib/python3.13/site-packages/tqdm/notebook.py", line 171, in display
  File "/home/u12/uliseshmc/.conda/envs/delme/lib/python3.13/site-packages/traitlets/traitlets.py", line 716, in __set__
  File "/home/u12/uliseshmc/.conda/envs/delme/lib/python3.13/site-packages/traitlets/traitlets.py", line 706, in set
  File "/home/u12/uliseshmc/.conda/envs/delme/lib/python3.13/site-packages/traitlets/traitlets.py", line 1513, in _notify_trait
  File "/home/u12/uliseshmc/.conda/envs/delme/lib/python3.13/site-packages/ipywidgets/widgets/widget.py", line 700, in notify_change
  File "/home/u12/uliseshmc/.conda/envs/delme/lib/python3.13/si

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -782674.3570
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        0.02               8.58    1.04    3.55    0.69
Chimpanzee    root        0.02               9.54    0.93    3.24    0.59
Gorilla       root        0.03               9.17    1.06    4.30    0.68
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
1.23    1.69    4.35    4.81    1.36    1.24    0.69    3.54
1.03    1.34    4.02    4.07    1.47    0.97    0.67    3.15
1.15    1.49    4.72    4.46    1.72    1.35    0.63    4.17
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.02    0.01    0.02    0.01    0.02    0.01    0.00    0.01    0.02    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.03    0.01    0.01    0.01    0.02    0.01    0.01    0.02    0.03    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.03    0.03    0.01    0.03    0.00    0.01    0.01    0.00    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.03    0.02    0.02    0.01    0.02    0.01    0.02    0.02    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.02    0.02    0.03    0.02    0.01    0.01    0.02    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.01    0.02    0.02    0.00    0.02    0.02    0.02    0.03    0.02
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.01    0.02    0.01    0.02
----------------------------

## Intergenic proximal 3' (5kb proximal from transcript end)

In [12]:
folder_in = paths.DATA_APES114 + 'proximal3_IG/'
in_dstore = cogent3.open_data_store(folder_in, suffix='fa', mode='r')

# noncds_app is defined on the Intergenic Ancestral Repeat section
nonconcat_proximal3IG = [r for r in noncds_app.as_completed(in_dstore[:], parallel=True) if r]
# This clear progress bar and avoids error messages
tqdm._instances.clear()

alns_proximal3IG = concat(nonconcat_proximal3IG)
alns_proximal3IG.source = "proximal3IG_alignments"

# The GT_subsmodel is defined on the Intergenic Ancestral Repeat section
result_proximal3IG = GT_subsmodel(alns_proximal3IG)
# This clear progress bar and avoids error messages
tqdm._instances.clear()

with open("fitted_models/likelihood_proximal3IG.pickle", mode = "wb") as out: 
    out.write(pickle.dumps(result_proximal3IG))

result_proximal3IG.lf

   0%|          |00:00<?

   0%|          |00:00<?

GT_CpG_ss
log-likelihood = -784652.7021
number of free parameters = 102
=====
omega
-----
 1.00
-----
=========================================================================
edge          parent    length    (CG>TG | CG>CA)     A>C     A>G     A>T
-------------------------------------------------------------------------
Human         root        0.02               8.26    1.03    3.91    0.83
Chimpanzee    root        0.02               9.40    0.80    3.39    0.64
Gorilla       root        0.03               8.40    0.96    4.52    0.72
-------------------------------------------------------------------------

continued: 
============================================================
 C>A     C>G     C>T     G>A     G>C     G>T     T>A     T>C
------------------------------------------------------------
1.28    1.52    4.82    4.74    1.55    1.37    0.63    4.26
1.03    1.31    3.70    3.53    1.29    0.85    0.60    2.96
1.20    1.42    4.36    4.57    1.61    1.42    0.67    4.41
------------------------------------------------------------

============================================================================
 AAA     AAC     AAG     AAT     ACA     ACC     ACG     ACT     AGA     AGC
----------------------------------------------------------------------------
0.02    0.01    0.02    0.01    0.02    0.02    0.00    0.01    0.02    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 AGG     AGT     ATA     ATC     ATG     ATT     CAA     CAC     CAG     CAT
----------------------------------------------------------------------------
0.03    0.01    0.01    0.01    0.01    0.01    0.01    0.02    0.03    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CCA     CCC     CCG     CCT     CGA     CGC     CGG     CGT     CTA     CTC
----------------------------------------------------------------------------
0.03    0.03    0.01    0.03    0.00    0.01    0.01    0.00    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 CTG     CTT     GAA     GAC     GAG     GAT     GCA     GCC     GCG     GCT
----------------------------------------------------------------------------
0.03    0.02    0.02    0.01    0.02    0.01    0.02    0.02    0.01    0.02
----------------------------------------------------------------------------

continued: 
============================================================================
 GGA     GGC     GGG     GGT     GTA     GTC     GTG     GTT     TAA     TAC
----------------------------------------------------------------------------
0.02    0.02    0.03    0.01    0.01    0.01    0.02    0.01    0.01    0.01
----------------------------------------------------------------------------

continued: 
============================================================================
 TAG     TAT     TCA     TCC     TCG     TCT     TGA     TGC     TGG     TGT
----------------------------------------------------------------------------
0.01    0.01    0.02    0.02    0.00    0.02    0.02    0.02    0.03    0.02
----------------------------------------------------------------------------

continued: 
============================
 TTA     TTC     TTG     TTT
----------------------------
0.01    0.02    0.01    0.02
----------------------------